# 06_finetune_marian_en_hi

In [7]:
from transformers import MarianMTModel, MarianTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-hi").to(device)


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [9]:
import sys
sys.path.append("/kaggle/input/datasets/kunjagarwal/en-hi-dataset")

In [10]:
import shutil
import os

src_path = "/kaggle/input/datasets/kunjagarwal/en-hi-dataset/data/processed/iitb_filtered"  # confirm this path is correct first
dst_path = "/kaggle/working/iitb_filtered"

shutil.copytree(src_path, dst_path)

from datasets import load_from_disk
dataset= load_from_disk(dst_path)
print(dataset)

FileExistsError: [Errno 17] File exists: '/kaggle/working/iitb_filtered'

In [12]:
SUBSET_SIZE = 180_000  # adjust as needed

train_subset = dataset_filtered["train"].shuffle(seed=42).select(range(SUBSET_SIZE))

In [14]:
val_subset = dataset_filtered["validation"].shuffle(seed=42).select(range(min(10_000, len(dataset_filtered["validation"]))))
test_subset = dataset_filtered["test"].shuffle(seed=42).select(range(min(10_000, len(dataset_filtered["test"]))))

In [15]:
def tokenize_batch(sources, targets, tokenizer, max_length=80):
    """
    Tokenizes a batch of source (English) and target (Hindi) sentences
    for MarianMT training. Returns input_ids, attention_mask, and labels
    with padding positions in labels set to -100 (so loss ignores them).
    """
    model_inputs = tokenizer(
        sources,
        text_target=targets,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    # Replace pad_token_id with -100 in labels, so CrossEntropyLoss ignores padding
    labels = model_inputs["labels"]
    labels[labels == tokenizer.pad_token_id] = -100
    model_inputs["labels"] = labels

    return model_inputs

In [16]:
sample_sources = ["Hello, how are you?", "This is a test."]
sample_targets = ["नमस्ते, आप कैसे हैं?", "यह एक परीक्षण है।"]

batch = tokenize_batch(sample_sources, sample_targets, tokenizer)

print(batch["input_ids"])
print(batch["attention_mask"])
print(batch["labels"])

tensor([[12110,     2,   287,    54,    27,    22,     0, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949],
        [  239,    23,    19,  2564,     3,     0, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 

In [17]:
from torch.utils.data import DataLoader, Dataset

class TranslationDataset(Dataset):
    def __init__(self, sources, targets, tokenizer, max_length=80):
        self.sources = sources
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        source = self.sources[idx]
        target = self.targets[idx]
        tokenized = tokenize_batch([source], [target], self.tokenizer, self.max_length)
        return {key: val.squeeze(0) for key, val in tokenized.items()}

In [18]:
train_dataset = TranslationDataset(train_subset["en"], train_subset["hi"], tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [19]:
val_dataset = TranslationDataset(val_subset["en"], val_subset["hi"], tokenizer)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [20]:
batch = next(iter(train_dataloader))
print(batch["input_ids"].shape)

torch.Size([16, 80])


In [21]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=3e-5)

In [22]:
import time

start = time.time()
for i, batch in enumerate(train_dataloader):
    if i >= 20:
        break
    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

elapsed = time.time() - start
per_batch = elapsed / 20
total_batches = len(train_dataloader)
print(f"{per_batch:.3f}s/batch, estimated epoch time: {(per_batch * total_batches)/60:.1f} min")

0.304s/batch, estimated epoch time: 56.9 min


In [23]:
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for i, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"  Batch {i}/{len(dataloader)} — loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

In [24]:
@torch.no_grad()
def evaluate_epoch(model, dataloader, device):
    model.eval()
    total_loss = 0.0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()

    return total_loss / len(dataloader)

In [25]:
NUM_EPOCHS = 2

for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    train_loss = train_epoch(model, train_dataloader, optimizer, device)
    val_loss = evaluate_epoch(model, val_dataloader, device)

    print(f"Train loss: {train_loss:.4f}, Validation loss: {val_loss:.4f}")

Epoch 1/2
  Batch 0/11250 — loss: 4.3780
  Batch 50/11250 — loss: 4.0461
  Batch 100/11250 — loss: 3.7727
  Batch 150/11250 — loss: 3.4215
  Batch 200/11250 — loss: 2.9022
  Batch 250/11250 — loss: 4.0656
  Batch 300/11250 — loss: 2.5388
  Batch 350/11250 — loss: 4.1200
  Batch 400/11250 — loss: 3.0967
  Batch 450/11250 — loss: 2.7904
  Batch 500/11250 — loss: 3.4332
  Batch 550/11250 — loss: 2.6000
  Batch 600/11250 — loss: 3.8412
  Batch 650/11250 — loss: 3.1615
  Batch 700/11250 — loss: 2.5092
  Batch 750/11250 — loss: 3.8511
  Batch 800/11250 — loss: 3.4014
  Batch 850/11250 — loss: 2.8133
  Batch 900/11250 — loss: 3.2706
  Batch 950/11250 — loss: 2.9693
  Batch 1000/11250 — loss: 3.4432
  Batch 1050/11250 — loss: 3.2123
  Batch 1100/11250 — loss: 2.6298
  Batch 1150/11250 — loss: 2.9542
  Batch 1200/11250 — loss: 3.4099
  Batch 1250/11250 — loss: 3.2219
  Batch 1300/11250 — loss: 2.2310
  Batch 1350/11250 — loss: 1.9603
  Batch 1400/11250 — loss: 4.0545
  Batch 1450/11250 — loss: 

In [26]:
predictions = []
references = []

for i in range(len(test_subset["en"])):
    sentence = test_subset["en"][i]
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True).to(device)
    translated = model.generate(**inputs, max_length=80, num_beams=4, early_stopping=True)
    translated_text = tokenizer.batch_decode(translated, skip_special_tokens=True)[0]
    predictions.append(translated_text)
    references.append(test_subset["hi"][i])

In [29]:
print(test_subset["en"][0])
print(predictions[0])
print(references[0])

They never believed he died the way the sheriff concluded.
उन्होंने कभी विश्वास नहीं किया कि वह इस तरह की मृत्यु हुई जिस तरह से प्रधान की मृत्यु हुई।
उन्हें शेरिफ के बताये गये तरीके से मृत्यु पर बिलकुल भी भरोसा नहीं है।


In [31]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 4.5 MB/s eta 0:00:00


In [32]:
import sacrebleu
bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"Fine-tuned EN→HI BLEU: {bleu.score:.2f}")

Fine-tuned EN→HI BLEU: 13.70


In [34]:
model.save_pretrained("models/marian_en_hi_finetuned")
tokenizer.save_pretrained("models/marian_en_hi_finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/marian_en_hi_finetuned/tokenizer_config.json',
 'models/marian_en_hi_finetuned/vocab.json',
 'models/marian_en_hi_finetuned/source.spm',
 'models/marian_en_hi_finetuned/target.spm',
 'models/marian_en_hi_finetuned/added_tokens.json')

In [36]:
import json

marian_finetuned_en_hi_results = {
    "model": "MarianMT EN-HI, fine-tuned (2 epochs, manual loop)",
    "bleu_score": bleu.score,
    "baseline_bleu_score": 9.85,  # Day 5's zero-shot number
}

with open("marian_finetuned_en_hi_results.json", "w") as f:
    json.dump(marian_finetuned_en_hi_results, f, indent=2, ensure_ascii=False)

In [37]:
import shutil

shutil.make_archive("download_files", "zip", "/kaggle/working/marian_en_hi_finetuned")

'/kaggle/working/download_files.zip'